In [8]:
import numpy as np
import pandas as pd
from itertools import product
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, classification_report, confusion_matrix
from xgboost import XGBRegressor, XGBClassifier

# 1. Load Data
items = pd.read_csv('items.csv')
shops = pd.read_csv('shops.csv')
cats = pd.read_csv('item_categories.csv')
train = pd.read_csv('sales_train.csv')
test  = pd.read_csv('test.csv').set_index('ID')

# 2. Preprocessing
train = train[train.item_price < 100000]
train = train[train.item_cnt_day < 1001]
train.loc[train.shop_id == 0, 'shop_id'] = 57
test.loc[test.shop_id == 0, 'shop_id'] = 57

shops['city'] = shops['shop_name'].str.split(' ').map(lambda x: x[0])
shops['city_code'] = LabelEncoder().fit_transform(shops['city'])
shops = shops[['shop_id','city_code']]

cats['type_code'] = LabelEncoder().fit_transform(cats['item_category_name'].str.split('-').map(lambda x: x[0].strip()))
cats = cats[['item_category_id','type_code']]

# 3. Aggregate Monthly Sales
matrix = []
for i in range(34):
    sales = train[train.date_block_num==i]
    if len(sales) > 0:
        matrix.append(np.array(list(product([i], sales.shop_id.unique(), sales.item_id.unique())), dtype='int16'))

matrix = pd.DataFrame(np.vstack(matrix), columns=['date_block_num','shop_id','item_id'])

group = train.groupby(['date_block_num','shop_id','item_id']).agg({'item_cnt_day': 'sum'})
group.columns = ['item_cnt_month']
matrix = pd.merge(matrix, group.reset_index(), on=['date_block_num','shop_id','item_id'], how='left').fillna(0)
matrix['item_cnt_month'] = matrix['item_cnt_month'].clip(0,20)

test['date_block_num'] = 34
matrix = pd.concat([matrix, test], ignore_index=True, sort=False).fillna(0)

# 4. Feature Engineering
matrix = pd.merge(matrix, shops, on=['shop_id'], how='left')
matrix = pd.merge(matrix, items.drop(['item_name'], axis=1), on=['item_id'], how='left')
matrix = pd.merge(matrix, cats, on=['item_category_id'], how='left')

# 5. Split Dataset
X_train = matrix[matrix.date_block_num < 33].drop(['item_cnt_month'], axis=1)
Y_train = matrix[matrix.date_block_num < 33]['item_cnt_month']
X_valid = matrix[matrix.date_block_num == 33].drop(['item_cnt_month'], axis=1)
Y_valid = matrix[matrix.date_block_num == 33]['item_cnt_month']
X_test = matrix[matrix.date_block_num == 34].drop(['item_cnt_month'], axis=1)

# 6. Regression Model (Exercise 1)
model_reg = XGBRegressor(n_estimators=1000, max_depth=8, learning_rate=0.03,
                         min_child_weight=300, colsample_bytree=0.8, subsample=0.8,
                         eval_metric="rmse", early_stopping_rounds=10, random_state=42)

model_reg.fit(X_train, Y_train, eval_set=[(X_valid, Y_valid)], verbose=False)
rmse = np.sqrt(mean_squared_error(Y_valid, model_reg.predict(X_valid)))
print(f'Validation RMSE: {rmse:.4f}')

pd.DataFrame({"ID": test.index, "item_cnt_month": model_reg.predict(X_test).clip(0,20)}).to_csv('submission_regression.csv', index=False)

# 7. Classification Model (Exercise 2) - LabelEncoding applied
def categorize_sales(x):
    if x == 0: return 'no_sales'
    elif x <= 5: return 'low'
    elif x <= 15: return 'medium'
    else: return 'high'

# Convert to numeric labels for XGBoost
le = LabelEncoder()
Y_train_cls = le.fit_transform(Y_train.apply(categorize_sales))
Y_valid_cls = le.transform(Y_valid.apply(categorize_sales))

model_cls = XGBClassifier(n_estimators=100, max_depth=8, random_state=42)
model_cls.fit(X_train, Y_train_cls)

Y_pred_cls = model_cls.predict(X_valid)
print("\n--- Classification Report ---")
print(classification_report(Y_valid_cls, Y_pred_cls, target_names=le.classes_))

Validation RMSE: 1.0074

--- Classification Report ---
              precision    recall  f1-score   support

        high       0.50      0.17      0.26       358
         low       0.55      0.12      0.19     29869
      medium       0.32      0.03      0.06      1244
    no_sales       0.88      0.99      0.93    206701

    accuracy                           0.87    238172
   macro avg       0.57      0.33      0.36    238172
weighted avg       0.84      0.87      0.83    238172

